# ✏️ Doodle-to-Emoji — train your own model

**In Google Colab, select Runtime → Run all once.** This notebook installs its
dependencies, streams a small Quick Draw training set, trains your three-class
CNN, converts and loads it, then opens the Gradio drawing app. The final cell
automatically enables a temporary public Gradio link in Colab. Keep the runtime
connected while using the app. A CPU runtime is sufficient; a GPU is optional.

The model learns **apple 🍎, star ⭐, and sun ☀️** from scratch. It replaces the
earlier pretrained classifier. There is no pretrained download or manual
reload step. The browser's Gradio Sketchpad provides the drawing canvas;
Python handles preprocessing and prediction on the notebook runtime.

The first run needs internet access and takes a few minutes for installation,
data loading, and training. Later Run all passes reuse the cached training
images and train a new model. Colab's temporary files disappear when its runtime
is deleted, so download the notebook/model if you want to keep them.


## 1. Install dependencies

Run this before the imports. TensorFlow trains and exports the CNN; LiteRT loads
the exported model; Gradio provides the web interface. These versions have
Python 3.12 and 3.13 Linux wheels for Colab.


In [17]:
%pip install -q "tensorflow==2.20.0" "gradio==5.49.1" "ai-edge-litert==2.2.0" "numpy>=1.26,<2.3" "Pillow>=10,<13"


## 2. Imports and settings

The defaults need no edits. To add a category later, add its Quick Draw name and
an emoji to `CATEGORY_EMOJIS`, then Run all again. Label order is saved with the
model, so the app uses exactly the vocabulary it was trained on.


In [18]:
import hashlib
import html
import json
import os
import sys
import threading
import time
import urllib.request
from pathlib import Path
from urllib.parse import quote

os.environ.setdefault("GRADIO_ANALYTICS_ENABLED", "False")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from ai_edge_litert.interpreter import Interpreter

CATEGORY_EMOJIS = {"apple": "🍎", "star": "⭐", "sun": "☀️"}
CATEGORIES = list(CATEGORY_EMOJIS)
SAMPLES_PER_CATEGORY = 2000
EPOCHS = 5
BATCH_SIZE = 32
SEED = 42
CANVAS_SIZE = 384

assert len(CATEGORIES) >= 2
assert SAMPLES_PER_CATEGORY >= 20
tf.keras.utils.set_random_seed(SEED)

# Keep custom weights separate from the earlier pretrained model cache.
MODEL_DIR = Path("work/custom_doodle_model")
DATA_DIR = Path("work/quickdraw_training")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "model.tflite"
labels_path = MODEL_DIR / "labels.txt"
print(f"Training {len(CATEGORIES)} classes: {', '.join(CATEGORIES)}")


Training 3 classes: apple, star, sun


## 3. Prepare drawings and load a small training set

Training sketches and user drawings use the **same** crop, centering, polarity,
and 28 × 28 normalization. Raw Quick Draw coordinates are scaled to fit the
canvas before rasterization, so large sketches are not clipped.

Only the first 2,000 usable recognized sketches per category are streamed from
the [official Quick Draw dataset](https://github.com/googlecreativelab/quickdraw-dataset).
The connection closes once enough examples have been read, avoiding full
hundreds-of-megabytes downloads. Processed images are cached for later runs.


In [19]:
def preprocess_sketch(value):
    """Return float32 [1, 28, 28, 1] and a PIL preview, or reject a blank canvas."""
    if isinstance(value, dict):  # Gradio Sketchpad sends an EditorValue dictionary.
        value = value.get("composite")
    if value is None:
        raise ValueError("Draw something first.")
    image = value if isinstance(value, Image.Image) else Image.fromarray(value)
    # Transparent pixels are empty paper; dropping alpha would turn them black.
    rgba = image.convert("RGBA")
    paper = Image.new("RGBA", rgba.size, "white")
    gray = Image.alpha_composite(paper, rgba).convert("L")
    pixels = np.asarray(gray)
    ink = pixels < 240
    ys, xs = np.where(ink)
    if len(xs) < 20 or max(int(np.ptp(xs)), int(np.ptp(ys))) < 8:
        raise ValueError("The canvas is empty or too small. Draw a larger sketch.")
    crop = gray.crop((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
    # Training and inference both use bright strokes on a dark background.
    crop = ImageOps.invert(crop)
    scale = 26 / max(crop.size)
    size = tuple(max(1, round(dimension * scale)) for dimension in crop.size)
    crop = crop.resize(size, Image.Resampling.LANCZOS)
    preview = Image.new("L", (28, 28), 0)
    preview.paste(crop, ((28 - size[0]) // 2, (28 - size[1]) // 2))
    tensor = np.asarray(preview, dtype=np.float32)[None, :, :, None] / 255.0
    return tensor, preview


In [20]:
def rasterize_training_drawing(drawing, line_width=8):
    strokes = [list(zip(stroke[0], stroke[1])) for stroke in drawing if len(stroke) >= 2]
    points = [point for stroke in strokes for point in stroke]
    if not points:
        raise ValueError("Empty training drawing")
    coordinates = np.asarray(points, dtype=np.float32)
    lower = coordinates.min(axis=0)
    extent = coordinates.max(axis=0) - lower
    span = float(extent.max())
    if span <= 0 or not np.all(np.isfinite(coordinates)):
        raise ValueError("Invalid training drawing")
    scale = (CANVAS_SIZE - 64) / span
    offset = (CANVAS_SIZE - extent * scale) / 2
    image = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), "white")
    pen = ImageDraw.Draw(image)
    for stroke in strokes:
        normalized = [tuple((np.asarray(point) - lower) * scale + offset) for point in stroke]
        if len(normalized) >= 2:
            pen.line(normalized, fill="black", width=line_width, joint="curve")
        elif normalized:
            x, y = normalized[0]
            radius = line_width / 2
            pen.ellipse((x - radius, y - radius, x + radius, y + radius), fill="black")
    _, preview = preprocess_sketch(image)
    return np.asarray(preview, dtype=np.uint8)


def load_training_data():
    # Changing classes, sample count, or preprocessing creates a new cache key.
    config = {"categories": CATEGORIES, "count": SAMPLES_PER_CATEGORY, "preprocessing": 1}
    key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
    cache = DATA_DIR / f"images_{key}.npz"
    if cache.exists():
        with np.load(cache, allow_pickle=False) as saved:
            images, targets = saved["images"], saved["targets"]
        print(f"Loaded {len(targets):,} cached training images.")
    else:
        images, targets = [], []
        for class_index, category in enumerate(CATEGORIES):
            url = f"https://storage.googleapis.com/quickdraw_dataset/full/raw/{quote(category, safe='')}.ndjson"
            print(f"Streaming {category}...", flush=True)
            accepted = 0
            with urllib.request.urlopen(url, timeout=120) as response:
                for line in response:
                    record = json.loads(line)
                    if not record.get("recognized", False):
                        continue
                    try:
                        # A few stroke widths help match different browser brush sizes.
                        image = rasterize_training_drawing(record["drawing"], 6 + 2 * (accepted % 4))
                    except (ValueError, KeyError, IndexError):
                        continue
                    images.append(image)
                    targets.append(class_index)
                    accepted += 1
                    if accepted == SAMPLES_PER_CATEGORY:
                        break
            if accepted < SAMPLES_PER_CATEGORY:
                raise RuntimeError(f"Only found {accepted} usable {category} sketches.")
            print(f"  Prepared {accepted:,} {category} sketches.", flush=True)
        images = np.stack(images)
        targets = np.asarray(targets, dtype=np.int32)
        temporary = cache.with_suffix(".tmp.npz")
        np.savez_compressed(temporary, images=images, targets=targets)
        temporary.replace(cache)
    expected = SAMPLES_PER_CATEGORY * len(CATEGORIES)
    if images.shape != (expected, 28, 28) or targets.shape != (expected,):
        raise RuntimeError(f"Invalid dataset cache. Delete {cache} and Run all again.")
    return images[..., None].astype(np.float32) / 255.0, targets


X, y = load_training_data()
print(f"Dataset: {X.shape}, values {X.min():.1f}–{X.max():.1f}")


Streaming apple...
  Prepared 2,000 apple sketches.
Streaming star...
  Prepared 2,000 star sketches.
Streaming sun...
  Prepared 2,000 sun sketches.
Dataset: (6000, 28, 28, 1), values 0.0–1.0


## 4. Train the custom CNN

Use 90% of each category for training and 10% for validation. The CNN is
initialized from scratch. The reported validation accuracy measures held-out
Quick Draw sketches; it is not a guarantee for every user's drawings.


In [21]:
rng = np.random.default_rng(SEED)
train_indices, validation_indices = [], []
for class_index in range(len(CATEGORIES)):
    indices = rng.permutation(np.flatnonzero(y == class_index))
    validation_count = max(1, int(len(indices) * 0.1))
    validation_indices.extend(indices[:validation_count])
    train_indices.extend(indices[validation_count:])
train_indices = rng.permutation(train_indices)
validation_indices = np.asarray(validation_indices)
X_train, y_train = X[train_indices], y[train_indices]
X_val, y_val = X[validation_indices], y[validation_indices]

# Use a small data-loading thread pool so CPU-only notebook runtimes work too.
options = tf.data.Options()
options.threading.private_threadpool_size = 2
train_data = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(y_train), seed=SEED)
    .batch(BATCH_SIZE).with_options(options).prefetch(1)
)
validation_data = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(BATCH_SIZE).with_options(options).prefetch(1)
)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(16, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(len(CATEGORIES), activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_data, validation_data=validation_data, epochs=EPOCHS, verbose=2)
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.1%}")


Epoch 1/5
169/169 - 6s - 33ms/step - accuracy: 0.8494 - loss: 0.3837 - val_accuracy: 0.9100 - val_loss: 0.2294
Epoch 2/5
169/169 - 3s - 15ms/step - accuracy: 0.9243 - loss: 0.2020 - val_accuracy: 0.9450 - val_loss: 0.1643
Epoch 3/5
169/169 - 5s - 30ms/step - accuracy: 0.9481 - loss: 0.1381 - val_accuracy: 0.9550 - val_loss: 0.1353
Epoch 4/5
169/169 - 6s - 36ms/step - accuracy: 0.9626 - loss: 0.1091 - val_accuracy: 0.9683 - val_loss: 0.1042
Epoch 5/5
169/169 - 3s - 15ms/step - accuracy: 0.9720 - loss: 0.0832 - val_accuracy: 0.9583 - val_loss: 0.0986
Final validation accuracy: 95.8%


## 5. Export and immediately load the new model

Save the new model and its labels together, then create a **new interpreter**
from those files. The app below uses this interpreter. Nothing downloads or
restores the old pretrained model.


In [22]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_bytes = converter.convert()
model_path.write_bytes(tflite_bytes)
labels_path.write_text("\n".join(CATEGORIES) + "\n", encoding="utf-8")
(MODEL_DIR / "training_info.json").write_text(json.dumps({
    "categories": CATEGORIES,
    "samples_per_category": SAMPLES_PER_CATEGORY,
    "epochs": EPOCHS,
    "seed": SEED,
    "validation_accuracy": float(history.history["val_accuracy"][-1]),
}, indent=2), encoding="utf-8")
print(f"Saved custom model: {model_path} ({len(tflite_bytes) / 1024:.0f} KiB)")


Saved artifact at '/tmp/tmpw8erztp7'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_8')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135808576344848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576348880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576345616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576350032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576352720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576348496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576355216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135808576350416: TensorSpec(shape=(), dtype=tf.resource, name=None)
Saved custom model: work/custom_doodle_model/model.tflite (224 KiB)


In [23]:
LABELS = labels_path.read_text(encoding="utf-8").splitlines()
assert LABELS == CATEGORIES, "The saved labels must match training order."
EMOJI_MAP = {label: CATEGORY_EMOJIS[label] for label in LABELS}
interpreter = Interpreter(model_path=str(model_path), num_threads=2)
interpreter.allocate_tensors()
INPUT = interpreter.get_input_details()[0]
OUTPUT = interpreter.get_output_details()[0]
MODEL_LOCK = threading.Lock()
assert tuple(INPUT["shape"]) == (1, 28, 28, 1)
assert tuple(OUTPUT["shape"]) == (1, len(LABELS))
print(f"App loaded the custom model with labels: {LABELS}")


App loaded the custom model with labels: ['apple', 'star', 'sun']


## 6. Prediction callbacks and checks

Verify that the exported model's predictions agree with the trained TensorFlow
model. The example star then exercises the same callback as the web app.


In [24]:
def model_probabilities(tensor):
    """Respect the TFLite input/output quantization scales and zero points."""
    dtype = INPUT["dtype"]
    if np.issubdtype(dtype, np.integer):
        scale, zero_point = INPUT["quantization"]
        limits = np.iinfo(dtype)
        model_input = np.clip(np.rint(tensor / scale + zero_point), limits.min, limits.max).astype(dtype)
    else:
        model_input = tensor.astype(dtype)
    # An Interpreter cannot process simultaneous calls safely.
    with MODEL_LOCK:
        interpreter.set_tensor(INPUT["index"], model_input)
        interpreter.invoke()
        output = interpreter.get_tensor(OUTPUT["index"])[0].copy()
    probabilities = output.astype(np.float32)
    if np.issubdtype(OUTPUT["dtype"], np.integer):
        scale, zero_point = OUTPUT["quantization"]
        probabilities = (probabilities - zero_point) * scale
    if not np.all(np.isfinite(probabilities)) or probabilities.sum() <= 0:
        raise RuntimeError("The model returned invalid scores.")
    return probabilities


def rank_emojis(probabilities, top_k=3):
    """Sum scores for categories sharing an emoji; keep unsupported mass unassigned."""
    grouped = {}
    for label, probability in zip(LABELS, probabilities):
        emoji = EMOJI_MAP.get(label.lower())
        score = float(probability)
        if emoji is None or score <= 0:
            continue
        if emoji not in grouped:
            grouped[emoji] = {"emoji": emoji, "label": label, "score": 0.0, "strongest": -1.0}
        item = grouped[emoji]
        item["score"] += score
        if score > item["strongest"]:
            item.update(label=label, strongest=score)
    # Do not renormalize over mapped labels: that would inflate weak matches.
    return sorted(grouped.values(), key=lambda item: item["score"], reverse=True)[:top_k]


In [25]:
READY_HTML = '<div class="emoji-result">✏️</div><p class="result-caption">Your emoji appears here</p>'


def predict_emoji(value):
    start = time.perf_counter()
    try:
        tensor, preview = preprocess_sketch(value)
    except ValueError as exc:
        return READY_HTML, [], str(exc), None, ""
    probabilities = model_probabilities(tensor)
    matches = rank_emojis(probabilities)
    raw_index = int(probabilities.argmax())
    raw_label = LABELS[raw_index]
    raw_score = float(probabilities[raw_index])
    if not matches:
        return READY_HTML, [], f"I see '{raw_label}', but it has no supported emoji match.", preview, ""
    best = matches[0]
    heading = "Closest supported emoji" if best["score"] >= 0.60 else "Tentative emoji suggestion"
    result = (
        f'<div class="emoji-result">{html.escape(best["emoji"])}</div>'
        f'<p class="result-caption">{heading}: {html.escape(best["label"])}</p>'
    )
    rows = [[item["emoji"], item["label"], f'{item["score"]:.1%}'] for item in matches]
    status = f"Leading sketch category: {raw_label} ({raw_score:.1%})."
    if raw_label.lower() not in EMOJI_MAP:
        status += " That category has no emoji mapping; the suggestions are alternatives."
    if best["score"] < 0.60:
        status += " Try a larger, simpler outline."
    status += f" Prediction: {(time.perf_counter() - start) * 1000:.0f} ms."
    return result, rows, status, preview, best["emoji"]


def blank_canvas():
    return Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), "white")


def reset_canvas():
    # None resets the entire editor, including brush strokes and undo history.
    return None, READY_HTML, [], "Draw one object, then click Match emoji.", None, ""


def make_example_star():
    """A locally generated sketch for checking the complete pipeline."""
    image = blank_canvas()
    points = []
    for index in range(10):
        angle = -np.pi / 2 + index * np.pi / 5
        radius = 140 if index % 2 == 0 else 58
        points.append((192 + radius * np.cos(angle), 192 + radius * np.sin(angle)))
    ImageDraw.Draw(image).line(points + [points[0]], fill="black", width=12, joint="curve")
    return image


def show_example_star():
    image = make_example_star()
    return (image,) + tuple(predict_emoji(image))


In [26]:
# Check a few held-out images per class, rather than assuming any sketch is correct.
check_indices = np.concatenate([np.flatnonzero(y_val == i)[:3] for i in range(len(LABELS))])
check_images = X_val[check_indices]
keras_scores = model(check_images, training=False).numpy()
lite_scores = np.stack([model_probabilities(image[None, ...]) for image in check_images])
np.testing.assert_allclose(lite_scores, keras_scores, rtol=1e-4, atol=1e-5)
np.testing.assert_allclose(lite_scores.sum(axis=1), 1.0, atol=1e-5)
assert rank_emojis(lite_scores[0])[0]["label"] == LABELS[int(lite_scores[0].argmax())]
assert predict_emoji(blank_canvas())[1] == []
assert reset_canvas()[0] is None
print(f"Passed: TensorFlow/LiteRT agreement on {len(check_images)} held-out sketches, label mapping, and blank-canvas handling.")

star_result = predict_emoji(make_example_star())
print(f"Example star: {star_result[4]} — {star_result[2]}")
from IPython.display import display
display(star_result[3])


Passed: TensorFlow/LiteRT agreement on 9 held-out sketches, label mapping, and blank-canvas handling.
Example star: ⭐ — Leading sketch category: star (100.0%). Prediction: 4 ms.


## 7. Build the Gradio drawing app

Draw with your mouse, trackpad, or touchscreen. **Clear** resets the canvas,
undo history, predictions, and preview. The displayed categories come from
the new model's labels.


In [27]:
CSS = """
.emoji-result {font-size: 112px; text-align: center; line-height: 1.5;}
.result-caption {text-align: center; font-size: 18px;}
"""


def build_app():
    with gr.Blocks(title="Doodle-to-Emoji", theme=gr.themes.Soft(), css=CSS, analytics_enabled=False) as demo:
        gr.Markdown("# ✏️ Doodle-to-Emoji\nDraw one object and find its closest supported emoji.")
        gr.Markdown("Try **" + ", ".join(LABELS) + "**. Use a simple outline.")
        with gr.Row():
            with gr.Column():
                canvas = gr.Sketchpad(
                    value=blank_canvas(), type="pil", image_mode="RGBA",
                    canvas_size=(CANVAS_SIZE, CANVAS_SIZE), fixed_canvas=True,
                    height=450, label="Draw here", layers=False, transforms=(),
                    brush=gr.Brush(colors=["#000000"], color_mode="fixed", default_size=6),
                    format="png",
                )
                with gr.Row():
                    match_button = gr.Button("Match emoji", variant="primary")
                    # Clear locally so canvas reset does not wait for a Colab round trip.
                    clear_button = gr.ClearButton(canvas, value="Clear")
                    example_button = gr.Button("Try a star")
            with gr.Column():
                result = gr.HTML(READY_HTML)
                suggestions = gr.Dataframe(
                    headers=["Emoji", "Sketch category", "Model score"],
                    datatype=["str", "str", "str"], value=[], interactive=False,
                    label="Up to three suggestions",
                )
                emoji_text = gr.Textbox(label="Copy your emoji", show_copy_button=True, interactive=False)
                status = gr.Textbox(value="Draw one object, then click Match emoji.", label="Status", interactive=False)
                with gr.Accordion("What the model sees", open=False):
                    preview = gr.Image(label="Model input", type="pil", image_mode="L", height=180)
                gr.Markdown(
                    "Scores come from sketch classification; they are not calibrated accuracy. "
                    "This demo matches a fixed emoji vocabulary, so unsupported drawings can be misclassified."
                )
        outputs = [result, suggestions, status, preview, emoji_text]
        match_button.click(predict_emoji, inputs=canvas, outputs=outputs, concurrency_limit=1)
        # Reset to None first, then restore white paper. Repeating the same image
        # (or None) alone can leave the editor's internal strokes unchanged.
        clear_button.click(
            reset_canvas, inputs=[], outputs=[canvas] + outputs, queue=False,
        ).then(blank_canvas, inputs=[], outputs=canvas, queue=False)
        canvas.clear(lambda: reset_canvas()[1:], inputs=[], outputs=outputs, queue=False)
        # Predict from the generated image itself: the Sketchpad value is not
        # updated until the browser finishes loading it, so chaining on the
        # canvas input made the first click use stale (blank) data.
        example_button.click(
            show_example_star, inputs=[], outputs=[canvas] + outputs, concurrency_limit=1,
        )
        with gr.Accordion("Supported sketch categories", open=False):
            gr.Markdown(", ".join(f"{emoji} {label}" for label, emoji in sorted(EMOJI_MAP.items())))
    return demo


## 8. Launch — this is the last code cell

In Colab, the public Gradio URL is enabled automatically and is printed below.
Open it if the inline app does not appear. The link works while the Colab
runtime remains active. Locally, the app uses a localhost URL instead.

**Try a star → Clear → draw again → Match emoji.** No earlier cell needs to be
rerun to activate the trained model.


In [28]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if "demo" in globals():
    demo.close()
demo = build_app()
demo.launch(inline=True, share=IN_COLAB, show_error=True, prevent_thread_lock=True)


Closing server running on port: 7860
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1c0404b3ee20b1d598.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Save your work / hackathon notes

- **Stack:** Python + TensorFlow/Keras training + LiteRT inference + Pillow/NumPy
  image preparation + Gradio Blocks/Sketchpad interface. Colab runs the Python
  server; Gradio supplies the browser canvas and temporary sharing link.
- **Vocabulary:** only the categories listed in `CATEGORY_EMOJIS`. This is a
  sketch classifier followed by a category-to-emoji lookup, so it can choose
  one of those classes even when a drawing is unsupported.
- **Model files:** `work/custom_doodle_model/model.tflite`, `labels.txt`, and
  `training_info.json`. In Colab, use the Files sidebar to download these before
  ending the runtime. Download the notebook through File → Download → .ipynb.
- **Data:** Google's [Quick Draw dataset](https://github.com/googlecreativelab/quickdraw-dataset),
  shared under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
  This notebook selects recognized apple/star/sun sketches, rasterizes them,
  and normalizes them for training. Credit the dataset in your presentation.


## 9. Zero-shot CLIP upgrade (experimental) - steps 2-3 with test checkpoints

Needs `emoji_list.json` from the emoji-list step (upload it to Colab, or rebuild it). Run the cells in order and **read each checkpoint before moving on**. This section does not change the running app; wiring it into `predict_emoji` comes after the checkpoints look good.

**Testing mindset**
- Test one layer at a time: text embeddings first, then image preprocessing, then the full path. A bug found late is hard to place.
- You are looking for *patterns*, not single results. Ten wrong answers that share a cause matter more than one odd miss.
- Judge with top-5, not top-1. Doodles are ambiguous, and the demo shows several candidates.
- Be suspicious of "hub" emojis that win for many different drawings. They hurt more than a rare miss.
- Write down failures, fix the cause (name, prompt, filter, preprocessing), then re-run the same tests. Change one thing at a time.
- Test with drawings made the way a demo audience draws: fast, shaky, small, thick lines.
- Keep a short list of 10 drawings that reliably work. Those are your demo.

### Step 1 - build `emoji_list.json` (skip if the file already exists)
Creates the filtered, drawable emoji list (about 1,200 entries) and saves it next to the notebook. Download it from the Colab Files panel, or commit it to GitHub, so a session reset does not lose it.

In [ ]:
!pip install -q emoji
import os, re, json, random, emoji

FORCE_REBUILD = True  # set False once you are happy with the list
if os.path.exists("emoji_list.json") and not FORCE_REBUILD:
    print("emoji_list.json already exists - skipping rebuild.")
else:
    SKIN_TONE = re.compile("[\U0001F3FB-\U0001F3FF]")
    FLAG = re.compile("[\U0001F1E6-\U0001F1FF]|[\U000E0020-\U000E007F]")

    def keep(char, info):
        return (info["status"] == emoji.STATUS["fully_qualified"]
                and not SKIN_TONE.search(char) and not FLAG.search(char)
                and "\u200d" not in char and info["E"] <= 15)

    filtered = {c: i for c, i in emoji.EMOJI_DATA.items() if keep(c, i)}
    entries = [{"emoji": c, "name": i["en"].strip(":").replace("_", " "),
                "keywords": [a.strip(":").replace("_", " ") for a in i.get("alias", [])]}
               for c, i in filtered.items()]

    SKIP_WORDS = ("button", "keycap", "square", "circle", "arrow", "symbol", "o'clock", "-thirty")
    entries = [x for x in entries if not any(w in x["name"] for w in SKIP_WORDS)]

    COLORS = r"(red|orange|yellow|green|blue|light blue|purple|brown|black|white|grey|pink) "
    seen, deduped = set(), []
    for x in entries:
        key = re.sub(r"(smiling|grinning|beaming|squinting)\s*", "", x["name"]).strip()
        if key.endswith("heart"):
            if re.match(COLORS, key) and not key.startswith("red "):
                continue  # keep only the red heart; other colours look identical
            key = re.sub(COLORS, "", key)
        if key not in seen:
            seen.add(key)
            deduped.append(x)

    json.dump(deduped, open("emoji_list.json", "w", encoding="utf8"), ensure_ascii=False, indent=1)
    print("saved", len(deduped), "emojis")

d = json.load(open("emoji_list.json", encoding="utf8"))
print(len(d), d[0])
for x in random.sample(d, 10):
    print(x["emoji"], x["name"], x["keywords"])

In [ ]:
# Download the generated list (Colab only). Alternative: Files panel -> right-click -> Download.
try:
    from google.colab import files
    files.download("emoji_list.json")
except ImportError:
    pass

### Step 2a - load CLIP and the emoji list

In [ ]:
!pip install -q open_clip_torch
import json, os, numpy as np, torch, open_clip
from PIL import Image, ImageDraw

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32", pretrained="laion2b_s34b_b79k")
clip_tokenizer = open_clip.get_tokenizer("ViT-B-32")
clip_model = clip_model.to(device).eval()

assert os.path.exists("emoji_list.json"), "Run the Step 1 cell above to create emoji_list.json."
items = json.load(open("emoji_list.json", encoding="utf8"))
print(device, len(items), "emojis")

### Step 2b - build text embeddings (cached to `text_emb.npy`)

In [ ]:
TEMPLATES = [
    "a simple black and white doodle of a {}",
    "a hand-drawn line sketch of a {}",
    "a child's drawing of a {}",
    "a quick pencil sketch of {}",
]
# Emojis to drop after checkpoint review (fill in from the checkpoint output).
EXCLUDE = set()

def embed_texts(texts):
    with torch.no_grad():
        v = clip_model.encode_text(clip_tokenizer(texts).to(device))
        return torch.nn.functional.normalize(v, dim=-1)

DEDUPE_THRESHOLD = 0.97  # text-text cosine; lower = prune more aggressively
items = [x for x in json.load(open("emoji_list.json", encoding="utf8")) if x["emoji"] not in EXCLUDE]
vecs = []
for x in items:
    names = [x["name"]] + x.get("keywords", [])[:2]
    per = [embed_texts([t.format(n) for t in TEMPLATES]).mean(0) for n in names]
    vecs.append(torch.nn.functional.normalize(torch.stack(per).mean(0), dim=0))
text_emb = torch.stack(vecs)

# Greedy near-duplicate pruning: a doodle cannot tell these apart and they waste top-5 slots.
order = sorted(range(len(items)), key=lambda i: len(items[i]["name"]))
kept = []
for i in order:
    if not kept or float((text_emb[i] @ text_emb[kept].T).max()) < DEDUPE_THRESHOLD:
        kept.append(i)
kept.sort()
print("pruned", len(items) - len(kept), "near-duplicates")
items = [items[i] for i in kept]
text_emb = text_emb[kept]
np.save("text_emb.npy", text_emb.cpu().numpy())
print(text_emb.shape)

### Checkpoint 1 - text side
Look for:
- **Near-duplicate pairs (> 0.95):** a drawing cannot tell these apart. Add one of each pair to `EXCLUDE`, then re-run Step 2b.
- **Text queries:** "cat", "pizza", "house"... should return the matching emoji in the top 3. If most miss, fix the prompts or names before going further.
- **Hubs:** emojis that show up in the top 5 for many unrelated queries. Exclude or investigate them.

In [ ]:
sims = text_emb @ text_emb.T
sims.fill_diagonal_(-1)
nn_sim, nn_idx = sims.max(1)
pairs = [(i, int(j), float(s)) for i, (j, s) in enumerate(zip(nn_idx, nn_sim)) if s > 0.95 and i < int(j)]
print(len(pairs), "near-duplicate pairs (>0.95)")
for i, j, s in sorted(pairs, key=lambda p: -p[2])[:30]:
    print(f"{s:.3f}", items[i]["emoji"], items[i]["name"], "<->", items[j]["emoji"], items[j]["name"])

def search_text(q, k=5):
    s = (embed_texts([f"a simple black and white doodle of a {q}"]) @ text_emb.T)[0]
    return [(items[i]["emoji"], items[i]["name"], round(float(s[i]), 3)) for i in s.topk(k).indices.tolist()]

QUERIES = ["cat", "dog", "pizza", "house", "heart", "car", "tree", "sun", "guitar", "flower",
           "star", "apple", "fish", "umbrella", "bicycle", "moon", "key", "clock", "airplane", "cup"]
from collections import Counter
hub = Counter()
for q in QUERIES:
    res = search_text(q)
    hub.update(e for e, _, _ in res)
    print(q, "->", res)
print("\nPossible hubs (in top 5 of 3+ queries):", [(e, n) for e, n in hub.items() if n >= 3])

### Step 3 - embed the drawing and rank

In [ ]:
def prep_drawing(value, crop=True):
    """Square RGB image of the strokes on white, or None if the canvas is blank."""
    if isinstance(value, dict):
        value = value.get("composite")
    if value is None:
        return None
    img = value.convert("RGBA")
    img = Image.alpha_composite(Image.new("RGBA", img.size, "white"), img).convert("L")
    ink = np.array(img) < 200
    if ink.sum() < 30:
        return None
    if not crop:
        return img.convert("RGB")
    ys, xs = np.where(ink)
    box = img.crop((xs.min(), ys.min(), xs.max() + 1, ys.max() + 1))
    side = int(max(box.size) * 1.3)
    sq = Image.new("L", (side, side), 255)
    sq.paste(box, ((side - box.width) // 2, (side - box.height) // 2))
    return sq.convert("RGB")

def clip_rank(value, k=5, crop=True):
    img = prep_drawing(value, crop=crop)
    if img is None:
        return None
    with torch.no_grad():
        v = clip_model.encode_image(clip_preprocess(img).unsqueeze(0).to(device))
        v = torch.nn.functional.normalize(v, dim=-1)
    sims_ = (v @ text_emb.T)[0]
    share = torch.softmax(sims_ * 100, dim=0)  # relative confidence; raw cosines are all ~0.3
    top = sims_.topk(k)
    return [(items[i]["emoji"], items[i]["name"], round(float(s), 3), round(float(share[i]), 3))
            for s, i in zip(top.values, top.indices.tolist())]

### Checkpoint 2 - image side
Look for:
- **Blank canvas must return `None`.** If not, raise the ink threshold in `prep_drawing`.
- **Cropped vs uncropped:** cropping should usually improve the star's rank. If it does not, revisit the preprocessing.
- **Score gap:** if top-1 and top-5 scores are almost equal, the model is unsure; show "Tentative" in the UI for those.
- **Your own doodles:** draw 10-20 in the app, save them, and run them here. Note which emojis win wrongly and add the repeat offenders to `EXCLUDE`.

In [ ]:
print("blank:", clip_rank(blank_canvas()))
star = make_example_star()
print("star (crop)   :", clip_rank(star))
print("star (no crop):", clip_rank(star, crop=False))

def shape_test(draw_fn):
    im = blank_canvas(); draw_fn(ImageDraw.Draw(im)); return clip_rank(im)

print("circle:", shape_test(lambda d: d.ellipse((90, 90, 300, 300), outline="black", width=10)))
print("house :", shape_test(lambda d: (d.rectangle((110, 190, 280, 320), outline="black", width=10),
                                       d.polygon([(90, 190), (195, 90), (300, 190)], outline="black", width=10))))
import math
def _heart(d):
    pts = [(195 + 10 * 16 * math.sin(t) ** 3,
            190 - 10 * (13 * math.cos(t) - 5 * math.cos(2 * t) - 2 * math.cos(3 * t) - math.cos(4 * t)))
           for t in np.linspace(0, 2 * math.pi, 80)]
    d.line(pts + [pts[0]], fill="black", width=10, joint="curve")
print("heart :", shape_test(_heart))

## 10. Cheap adapter + hybrid app (keeps the 3 trained classes)

Run section 9 first (it provides `items`, `text_emb`, `clip_model`, `prep_drawing`). This section:
1. generates **synthetic sketches** from emoji glyphs (line art + wobble + stroke dropout),
2. caches their frozen CLIP features,
3. trains a small **linear adapter** (starts as the identity, pulled back toward it by a regulariser),
4. evaluates it on **your own real doodles** and only turns it on if it beats zero-shot,
5. wires everything into the app: the original apple / star / sun CNN is used when it is confident
   and CLIP agrees, and CLIP covers every other emoji.

**Mindset:** the synthetic held-out score only shows the adapter fits; it does *not* prove it helps on real drawings. Trust the real-doodle evaluation (10.4). If there is no evidence, the notebook stays on zero-shot CLIP.

### 10.1 - synthetic sketch generator

In [ ]:
!apt-get -qq install -y fonts-noto-color-emoji > /dev/null 2>&1
import random
from PIL import ImageFont, ImageFilter, ImageChops

EMOJI_FONT_PATHS = [
    "/usr/share/fonts/truetype/noto/NotoColorEmoji.ttf",
    "NotoColorEmoji.ttf",
]
font_path = next((p for p in EMOJI_FONT_PATHS if os.path.exists(p)), None)
assert font_path, "Noto Color Emoji font not found (run this in Colab, or install the font)."
emoji_font = ImageFont.truetype(font_path, 109)  # bitmap emoji fonts only support size 109


def render_emoji(ch):
    """Colour glyph as an RGBA image cropped to its visible pixels, or None if the font lacks it."""
    canvas = Image.new("RGBA", (160, 160), (0, 0, 0, 0))
    ImageDraw.Draw(canvas).text((10, 10), ch, font=emoji_font, embedded_color=True)
    box = canvas.getchannel("A").point(lambda a: 255 if a > 20 else 0).getbbox()
    if box is None or min(box[2] - box[0], box[3] - box[1]) < 20:
        return None
    return canvas.crop(box)


def to_lineart(rgba, size=256):
    """Thin outline + inner edges as a float mask (1 = ink) on a square canvas."""
    side = max(rgba.size)
    sq = Image.new("RGBA", (side, side), (0, 0, 0, 0))
    sq.paste(rgba, ((side - rgba.width) // 2, (side - rgba.height) // 2))
    sq = sq.resize((size, size), Image.Resampling.LANCZOS)
    alpha = sq.getchannel("A").point(lambda a: 255 if a > 100 else 0)
    outline = ImageChops.subtract(alpha, alpha.filter(ImageFilter.MinFilter(3)))
    white = Image.alpha_composite(Image.new("RGBA", sq.size, "white"), sq).convert("L")
    edges = white.filter(ImageFilter.GaussianBlur(1)).filter(ImageFilter.FIND_EDGES)
    edges = edges.point(lambda v: 255 if v > 18 else 0)
    inside = alpha.filter(ImageFilter.MinFilter(5))
    edges = ImageChops.multiply(edges, inside)
    return np.asarray(ImageChops.lighter(outline, edges), dtype=np.float32) / 255.0


def augment_sketch(mask, rng, canvas=CANVAS_SIZE):
    """Make a clean line mask look like a quick human doodle; return an RGB image like the app canvas."""
    n = mask.shape[0]
    # wobble: smooth sinusoidal displacement of the sampling grid
    yy, xx = np.mgrid[0:n, 0:n].astype(np.float32)
    amp = rng.uniform(1.5, 6.0)
    fx, fy = rng.uniform(0.01, 0.04, 2)
    px, py = rng.uniform(0, 6.28, 2)
    sx = np.clip(xx + amp * np.sin(yy * fx * 6.28 + px), 0, n - 1).astype(int)
    sy = np.clip(yy + amp * np.sin(xx * fy * 6.28 + py), 0, n - 1).astype(int)
    img = Image.fromarray((mask[sy, sx] * 255).astype(np.uint8))
    img = img.rotate(rng.uniform(-18, 18), resample=Image.Resampling.BILINEAR)
    img = img.filter(ImageFilter.MaxFilter(int(rng.choice([3, 5, 7, 9]))))  # stroke thickness
    arr = np.asarray(img).copy()
    for _ in range(int(rng.integers(0, 4))):  # dropped / missing strokes
        w, h = rng.integers(15, 70, 2)
        x0, y0 = rng.integers(0, n - w), rng.integers(0, n - h)
        arr[y0:y0 + h, x0:x0 + w] = 0
    ink = Image.fromarray(255 - arr)  # black ink on white
    scale = rng.uniform(0.35, 0.95)
    side = max(8, int(canvas * scale))
    ink = ink.resize((side, side), Image.Resampling.LANCZOS)
    page = Image.new("L", (canvas, canvas), 255)
    page.paste(ink, (int(rng.integers(0, canvas - side + 1)), int(rng.integers(0, canvas - side + 1))))
    return page.convert("RGB")


# quick visual check on a few emojis
_rng = np.random.default_rng(0)
_demo = []
for _ch in ["\U0001F431", "\U0001F355", "\U0001F3E0", "\U0001F697"]:
    _g = render_emoji(_ch)
    if _g is not None:
        _m = to_lineart(_g)
        _demo += [augment_sketch(_m, _rng).resize((160, 160)) for _ in range(3)]
if _demo:
    _sheet = Image.new("RGB", (160 * 6, 160 * ((len(_demo) + 5) // 6)), "white")
    for _i, _im in enumerate(_demo):
        _sheet.paste(_im, ((_i % 6) * 160, (_i // 6) * 160))
    display(_sheet)

**Checkpoint 10.1 - look at the sheet above.** The strokes should resemble doodles of a cat, pizza, house and car. If they are blank, speckled or unrecognisable, fix the generator (edge threshold, thickness) *before* spending time on training. Garbage in, garbage out.

### 10.2 - embed the synthetic sketches (one-time, cached)

In [ ]:
AUG_PER_EMOJI = 6        # more = better coverage, slower embedding (use a GPU runtime if possible)
VAL_AUG = 1              # last N augmentations per emoji are held out for validation
GEN_SEED = 7

def embed_images(pil_images, batch=64):
    out = []
    for i in range(0, len(pil_images), batch):
        x = torch.stack([clip_preprocess(im) for im in pil_images[i:i + batch]]).to(device)
        with torch.no_grad():
            out.append(torch.nn.functional.normalize(clip_model.encode_image(x), dim=-1).cpu())
    return torch.cat(out)

synth_emoji, synth_aug, synth_images = [], [], []
rng = np.random.default_rng(GEN_SEED)
skipped = []
for x in items:
    glyph = render_emoji(x["emoji"])
    if glyph is None:
        skipped.append(x["emoji"])
        continue
    mask = to_lineart(glyph)
    for a in range(AUG_PER_EMOJI):
        img = prep_drawing(augment_sketch(mask, rng))
        if img is not None:
            synth_emoji.append(x["emoji"]); synth_aug.append(a); synth_images.append(img)
print(f"{len(synth_images)} sketches for {len(set(synth_emoji))} emojis; {len(skipped)} emojis had no glyph: {''.join(skipped[:30])}")

synth_feats = embed_images(synth_images)
torch.save({"feats": synth_feats, "emoji": synth_emoji, "aug": synth_aug}, "synth_feats.pt")
print(synth_feats.shape)

### 10.3 - train the linear adapter (identity start, regularised back toward identity)

In [ ]:
ADAPTER_EPOCHS = 40
ADAPTER_LR = 2e-4
ADAPTER_REG = 1e-3       # higher = stays closer to zero-shot CLIP
LOGIT_SCALE = 100.0

idx_of = {x["emoji"]: i for i, x in enumerate(items)}
keep = [j for j, e in enumerate(synth_emoji) if e in idx_of]
X = synth_feats[keep].to(device)
Y = torch.tensor([idx_of[synth_emoji[j]] for j in keep], device=device)
is_val = torch.tensor([synth_aug[j] >= AUG_PER_EMOJI - VAL_AUG for j in keep], device=device)
Xtr, Ytr, Xva, Yva = X[~is_val], Y[~is_val], X[is_val], Y[is_val]
T = text_emb.to(device)

def topk_acc(logits, y, k):
    return float((logits.topk(k, dim=1).indices == y[:, None]).any(1).float().mean())

torch.manual_seed(0)
adapter_model = torch.nn.Linear(512, 512, bias=False).to(device)
with torch.no_grad():
    adapter_model.weight.copy_(torch.eye(512))
eye = torch.eye(512, device=device)
opt = torch.optim.AdamW(adapter_model.parameters(), lr=ADAPTER_LR, weight_decay=0.0)

def adapted_logits(x):
    return LOGIT_SCALE * torch.nn.functional.normalize(adapter_model(x), dim=-1) @ T.T

with torch.no_grad():
    base = LOGIT_SCALE * Xva @ T.T
print(f"zero-shot  synthetic val: top1 {topk_acc(base, Yva, 1):.3f}  top5 {topk_acc(base, Yva, 5):.3f}")
for epoch in range(ADAPTER_EPOCHS):
    perm = torch.randperm(len(Xtr), device=device)
    for i in range(0, len(Xtr), 256):
        b = perm[i:i + 256]
        loss = torch.nn.functional.cross_entropy(adapted_logits(Xtr[b]), Ytr[b])
        loss = loss + ADAPTER_REG * (adapter_model.weight - eye).pow(2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 10 == 9 or epoch == ADAPTER_EPOCHS - 1:
        with torch.no_grad():
            va = adapted_logits(Xva)
            drift = float((adapter_model.weight - eye).norm() / eye.norm())
        print(f"epoch {epoch + 1:3d}  loss {float(loss):.3f}  val top1 {topk_acc(va, Yva, 1):.3f}  "
              f"top5 {topk_acc(va, Yva, 5):.3f}  drift from identity {drift:.3f}")
torch.save(adapter_model.state_dict(), "adapter.pt")

**Checkpoint 10.3 - what to look for**
- Synthetic val top-1/top-5 should rise above the zero-shot line. If it does not move, the generator or learning rate is the problem.
- **Drift** near 0 means the adapter barely changed anything; a large drift (> ~0.5) with a huge val jump usually means it is memorising synthetic quirks. Raise `ADAPTER_REG` and compare again.
- A big synthetic gain does **not** guarantee a real gain: synthetic line art is cleaner than human drawing. The next step decides.

### 10.3b - quick start: auto-fill `eval_doodles/` from Quick Draw (optional, ~2 min)

Streams a few real human doodles for categories that match your emojis and saves them as `cat.png`, `cat_2.png`... (named after the emoji), ready for 10.4. Add your own drawings to the same folder for a fairer test.

Notes: Quick Draw sketches are cleaner than a demo audience's drawings, so expect slightly flattering numbers. The adapter is trained only on synthetic sketches, so there is no leakage. apple / star / sun are skipped (the CNN already handles them and was trained on the start of that data). Categories whose download fails or whose emoji is not in your list are reported and skipped.

In [ ]:
PER_CATEGORY = 3
SKIP_FIRST = 100          # skip the first records so the doodles are not the same ones everyone sees first
EVAL_DIR = "eval_doodles"
os.makedirs(EVAL_DIR, exist_ok=True)

# Quick Draw category -> acceptable emoji names (first one found in your list is used)
QD_TO_EMOJI = {
    "cat": ["cat", "cat face"], "dog": ["dog", "dog face"], "pizza": ["pizza"], "house": ["house"],
    "car": ["automobile"], "tree": ["deciduous tree"], "guitar": ["guitar"], "flower": ["blossom", "tulip", "white flower"],
    "fish": ["fish"], "umbrella": ["umbrella"], "bicycle": ["bicycle"], "moon": ["crescent moon"], "key": ["key"],
    "clock": ["mantelpiece clock", "alarm clock"], "airplane": ["airplane"], "banana": ["banana"],
    "cow": ["cow", "cow face"], "pig": ["pig", "pig face"], "horse": ["horse", "horse face"],
    "rabbit": ["rabbit", "rabbit face"], "bird": ["bird"], "butterfly": ["butterfly"], "snake": ["snake"],
    "spider": ["spider"], "strawberry": ["strawberry"], "hot dog": ["hot dog"], "ice cream": ["ice cream", "soft ice cream"],
    "hamburger": ["hamburger"], "camera": ["camera"], "scissors": ["scissors"], "crown": ["crown"],
    "diamond": ["gem stone"], "skull": ["skull"], "snowman": ["snowman"], "rainbow": ["rainbow"], "cloud": ["cloud"],
    "bed": ["bed"], "bus": ["bus"], "train": ["train"], "truck": ["delivery truck"], "sailboat": ["sailboat"],
    "tooth": ["tooth"], "eye": ["eye"], "bell": ["bell"], "candle": ["candle"], "cake": ["birthday cake"],
    "cookie": ["cookie"], "carrot": ["carrot"], "grapes": ["grapes"], "pear": ["pear"], "mushroom": ["mushroom"],
    "octopus": ["octopus"], "penguin": ["penguin"], "elephant": ["elephant"], "giraffe": ["giraffe"],
    "lion": ["lion"], "monkey": ["monkey", "monkey face"], "owl": ["owl"], "whale": ["whale", "spouting whale"],
    "shark": ["shark"], "frog": ["frog"], "bear": ["bear"], "pineapple": ["pineapple"],
    "watermelon": ["watermelon"], "broccoli": ["broccoli"], "donut": ["doughnut"], "envelope": ["envelope"],
    "hammer": ["hammer"], "anchor": ["anchor"], "balloon": ["balloon"], "basketball": ["basketball"],
    "violin": ["violin"], "trumpet": ["trumpet"], "piano": ["musical keyboard"], "lightning": ["high voltage"],
    "light bulb": ["light bulb"], "book": ["open book", "closed book"], "telephone": ["telephone receiver", "telephone"],
}


def render_quickdraw(drawing, line_width):
    strokes = [list(zip(s[0], s[1])) for s in drawing if len(s) >= 2 and len(s[0]) >= 2]
    pts = np.asarray([pt for s in strokes for pt in s], dtype=np.float32)
    lo = pts.min(0)
    extent = pts.max(0) - lo
    span = float(extent.max())
    if span <= 0:
        raise ValueError("empty")
    scale = (CANVAS_SIZE - 64) / span
    offset = (CANVAS_SIZE - extent * scale) / 2
    img = Image.new("RGB", (CANVAS_SIZE, CANVAS_SIZE), "white")
    pen = ImageDraw.Draw(img)
    for s in strokes:
        pen.line([tuple((np.asarray(pt) - lo) * scale + offset) for pt in s], fill="black",
                 width=line_width, joint="curve")
    return img


names_in_list = {x["name"] for x in items}
saved, problems = 0, []
for category, candidates in QD_TO_EMOJI.items():
    emoji_name = next((n for n in candidates if n in names_in_list), None)
    if emoji_name is None:
        problems.append(f"{category}: no matching emoji in your list")
        continue
    url = f"https://storage.googleapis.com/quickdraw_dataset/full/raw/{quote(category, safe='')}.ndjson"
    taken, seen = 0, 0
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            for line in response:
                record = json.loads(line)
                if not record.get("recognized", False):
                    continue
                seen += 1
                if seen <= SKIP_FIRST:
                    continue
                try:
                    img = render_quickdraw(record["drawing"], 6 + 3 * taken)
                except (ValueError, KeyError, IndexError):
                    continue
                suffix = "" if taken == 0 else f"_{taken + 1}"
                img.save(f"{EVAL_DIR}/{emoji_name}{suffix}.png")
                taken += 1
                saved += 1
                if taken == PER_CATEGORY:
                    break
    except Exception as exc:
        problems.append(f"{category}: download failed ({exc})")
print(f"Saved {saved} doodles to {EVAL_DIR}/")
for line in problems:
    print("  skipped", line)

### 10.4 - evaluate on your real doodles (decides whether the adapter is used)

Put 30+ of your own doodles in an `eval_doodles/` folder (Colab Files panel). Name each file after the emoji **name** from `emoji_list.json`, optionally with a suffix: `cat.png`, `cat_2.png`, `pizza.png`, `house_3.png`. Include different people's drawings, especially thick, shaky and small ones. **Never use these for training.**

In [ ]:
import glob, re as _re
ADAPTER_ALPHA = 0.0      # 0 = pure zero-shot CLIP; chosen below from your real doodles
adapter_ready = True

def encode_image(img):
    with torch.no_grad():
        v = clip_model.encode_image(clip_preprocess(img).unsqueeze(0).to(device))
        return torch.nn.functional.normalize(v, dim=-1)

def clip_scores(img, alpha=None):
    """Cosine scores over all emojis; blends zero-shot and adapter scores by alpha."""
    alpha = ADAPTER_ALPHA if alpha is None else alpha
    v = encode_image(img)
    s = (v @ text_emb.T)[0]
    if alpha > 0:
        with torch.no_grad():
            va = torch.nn.functional.normalize(adapter_model(v), dim=-1)
        s = (1 - alpha) * s + alpha * (va @ text_emb.T)[0]
    return s

name_to_idx = {x["name"]: i for i, x in enumerate(items)}
name_to_idx.update({x["emoji"]: i for i, x in enumerate(items)})
real = []
for f in sorted(glob.glob("eval_doodles/*.*")):
    key = _re.sub(r"_\d+$", "", os.path.splitext(os.path.basename(f))[0]).replace("-", " ")
    img = prep_drawing(Image.open(f))
    if key in name_to_idx and img is not None:
        real.append((f, name_to_idx[key], img))
    else:
        print("skipped (unknown name or blank):", f)

if len(real) < 10:
    print(f"Only {len(real)} usable real doodles in eval_doodles/. Staying on zero-shot CLIP (ADAPTER_ALPHA = 0).")
else:
    feats = torch.cat([encode_image(im) for _, _, im in real])
    truth = torch.tensor([i for _, i, _ in real], device=device)
    with torch.no_grad():
        adapted = torch.nn.functional.normalize(adapter_model(feats), dim=-1)
    results = {}
    for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
        s = (1 - a) * (feats @ T.T) + a * (adapted @ T.T)
        results[a] = (topk_acc(s, truth, 1), topk_acc(s, truth, 5))
        print(f"alpha {a:.2f}: top1 {results[a][0]:.3f}  top5 {results[a][1]:.3f}  ({len(real)} real doodles)")
    best = max(results, key=lambda a: (results[a][1], results[a][0]))
    # require a real gain over zero-shot, not noise
    if results[best][1] > results[0.0][1] + 0.02 or results[best][0] > results[0.0][0] + 0.03:
        ADAPTER_ALPHA = best
    print("ADAPTER_ALPHA =", ADAPTER_ALPHA, "(0.0 means the adapter did not clearly beat zero-shot)")

**Checkpoint 10.4 - how to read it**
- With few doodles, one or two drawings move top-1 by several points. The notebook demands a clear gain before enabling the adapter.
- If alpha 1.0 is *worse* than 0.0, the adapter overfits the synthetic style: raise `ADAPTER_REG`, lower `ADAPTER_EPOCHS`, or improve the generator, and re-run 10.3.
- Look at which emojis fail, not only the averages. Repeat offenders go in `EXCLUDE` (section 9).

### 10.5 - hybrid prediction and relaunch

Rules (constants are easy to tune):
- The original **CNN** answers for apple / star / sun when its probability is at least `CNN_MIN_PROB` **and** that emoji is also in CLIP's top `CNN_CLIP_TOPK` (this stops the 3-class CNN from confidently labelling, say, a cat as a sun).
- Otherwise **CLIP** (with the adapter if it won in 10.4) provides the ranking.
- "Closest" vs "Tentative" for CLIP uses its relative confidence (`CLIP_CONF`).

In [ ]:
CNN_MIN_PROB = 0.90
CNN_CLIP_TOPK = 10
CLIP_CONF = 0.35
SOFTMAX_TEMP = 100.0

predict_emoji_cnn = globals().get("predict_emoji_cnn", predict_emoji)  # original 3-class callback


def hybrid_rank(value, k=3):
    """Return (ranked list of dicts, source) or None if CLIP sees a blank canvas."""
    img = prep_drawing(value)
    if img is None:
        return None
    scores = clip_scores(img)
    share = torch.softmax(scores * SOFTMAX_TEMP, dim=0)
    top = scores.topk(max(k, CNN_CLIP_TOPK)).indices.tolist()
    ranked = [{"emoji": items[i]["emoji"], "label": items[i]["name"], "score": float(share[i])} for i in top]
    cnn_best = None
    try:
        tensor, _ = preprocess_sketch(value)
        matches = rank_emojis(model_probabilities(tensor))
        cnn_best = matches[0] if matches else None
    except ValueError:
        pass
    in_vocab = {x["emoji"] for x in items}
    if cnn_best and cnn_best["score"] >= CNN_MIN_PROB and (
            cnn_best["emoji"] not in in_vocab or cnn_best["emoji"] in [r["emoji"] for r in ranked[:CNN_CLIP_TOPK]]):
        rest = [r for r in ranked if r["emoji"] != cnn_best["emoji"]]
        return ([{"emoji": cnn_best["emoji"], "label": cnn_best["label"], "score": cnn_best["score"]}] + rest)[:k], "cnn", img
    return ranked[:k], "clip", img


def predict_emoji_hybrid(value):
    start = time.perf_counter()
    out = hybrid_rank(value)
    if out is None:
        return predict_emoji_cnn(value)  # reuses the original empty-canvas handling
    ranked, source, img = out
    best = ranked[0]
    threshold = 0.60 if source == "cnn" else CLIP_CONF
    heading = "Closest supported emoji" if best["score"] >= threshold else "Tentative emoji suggestion"
    result = (
        f'<div class="emoji-result">{html.escape(best["emoji"])}</div>'
        f'<p class="result-caption">{heading}: {html.escape(best["label"])}</p>'
    )
    rows = [[r["emoji"], r["label"], f'{r["score"]:.1%}'] for r in ranked]
    status = ("Answered by the trained apple/star/sun model." if source == "cnn"
              else "Answered by CLIP" + (" + adapter." if ADAPTER_ALPHA > 0 else " (zero-shot)."))
    if best["score"] < threshold:
        status += " Low confidence: try a larger, simpler outline."
    status += f" Prediction: {(time.perf_counter() - start) * 1000:.0f} ms."
    return result, rows, status, img.convert("L"), best["emoji"]


# checks: the three trained emojis still win, blank stays blank
assert predict_emoji_hybrid(blank_canvas())[1] == []
star_out = predict_emoji_hybrid(make_example_star())
print("hybrid star:", star_out[4], "-", star_out[2])
assert star_out[4] == "\u2b50", "the example star should still map to the star emoji"

predict_emoji = predict_emoji_hybrid  # build_app and show_example_star look this name up at call time
if "demo" in globals():
    demo.close()
demo = build_app()
demo.launch(inline=True, share=IN_COLAB, show_error=True, prevent_thread_lock=True)

### After the hybrid app is running - what to test
- **Try a star** must still give the star on the first click.
- Draw apple, star and sun: the status line should say they were answered by the trained model.
- Draw 10 other things (cat, pizza, house, car, key, umbrella, guitar...). Note which wins wrongly more than once.
- Draw something outside the vocabulary: the result should say "Tentative" rather than sound sure.
- Turn the adapter on and off (`ADAPTER_ALPHA`) on the same drawings. If you cannot see a difference, keep it off.